## Total Forecasting 

### 초반 설정 (패키지 설치)

In [2]:
%pip install pandas numpy holidays scikit-learn xgboost pymysql sshtunnel python-dotenv openpyxl requests tqdm urllib3 ipykernel 

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [7]:
!pip uninstall -y paramiko
!pip install paramiko==3.5.1 sshtunnel==0.4.0

Found existing installation: paramiko 3.5.1
Uninstalling paramiko-3.5.1:
  Successfully uninstalled paramiko-3.5.1
Defaulting to user installation because normal site-packages is not writeable
  Using cached paramiko-3.5.1-py3-none-any.whl.metadata (4.6 kB)
Using cached paramiko-3.5.1-py3-none-any.whl (227 kB)



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [4]:
import sys
print(sys.executable)

C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe


In [4]:
import pandas as pd
import pymysql
from sshtunnel import SSHTunnelForwarder
from dotenv import load_dotenv
import requests
import numpy as np
from tqdm import tqdm

print("import 성공")

import 성공


In [3]:
pip install "paramiko<4"

Defaulting to user installation because normal site-packages is not writeable
  Attempting uninstall: paramiko
    Found existing installation: paramiko 4.0.0
    Uninstalling paramiko-4.0.0:
      Successfully uninstalled paramiko-4.0.0
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.1
[notice] To update, run: C:\Users\딜리버스\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


### Data load & Update

In [1]:
# ==========================================================
# Cell 1. 데이터 로드 + 기온 병합 + 5-Track 타겟 생성
# ==========================================================

import os
import numpy as np
import pandas as pd
import pymysql
import requests
import urllib3

from dotenv import load_dotenv
from sshtunnel import SSHTunnelForwarder
from tqdm import tqdm
from datetime import datetime, timedelta

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
load_dotenv()


# ==========================================================
# 1. DB 데이터 로드
# ==========================================================
class AutoContainerGeneration:
    def __init__(self, version=7, debug=False):
        self.version = version
        self.debug = debug
        print(f"AutoContainerGeneration version: {version}")

    def fetch_data(
        self,
        query,
        ssh_host,
        ssh_user,
        ssh_private_key,
        mysql_host,
        mysql_port,
        mysql_user,
        mysql_password,
        mysql_database,
    ):
        try:
            with SSHTunnelForwarder(
                (ssh_host, 22),
                ssh_username=ssh_user,
                ssh_private_key=ssh_private_key,
                remote_bind_address=(mysql_host, mysql_port),
            ) as tunnel:
                print("SSH 터널 연결 성공")

                with pymysql.connect(
                    host="127.0.0.1",
                    user=mysql_user,
                    passwd=mysql_password,
                    db=mysql_database,
                    charset="utf8",
                    port=tunnel.local_bind_port,
                    cursorclass=pymysql.cursors.DictCursor,
                ) as conn:
                    with conn.cursor() as cur:
                        cur.execute(query)
                        results = cur.fetchall()
                        print("쿼리 실행 완료")
                        return pd.DataFrame(results)

        except Exception as e:
            print(f"Error fetching data: {e}")
            return None

    def fetch_cluster_data(self):
        ssh_host = os.getenv("SSH_HOST_VER_1")
        ssh_user = os.getenv("SSH_USER")
        ssh_private_key = os.getenv("SSH_PRIVATE_KEY")

        mysql_host = os.getenv("MYSQL_HOST")
        mysql_port = 3306
        mysql_user = os.getenv("MYSQL_USER")
        mysql_password = os.getenv("MYSQL_PASSWORD")
        mysql_database = os.getenv("MYSQL_DATABASE")

        print(f"Environment: {mysql_host}")

        # v.7 변경점:
        # - 지그재그는 당일/이월 유지
        # - 카카오모빌리티, 부스터스는 독립 track으로 추출
        # - 기타는 더 이상 기타_당일/기타_이월로 분리하지 않음
        # - 순수_기타물량은 Python에서 residual로 생성
        cluster_query = """
            SELECT
                hub_cluster_plan.plan_date AS 날짜,

                COUNT(DISTINCT shipping_shippingitem.id) AS "총 물량",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'sameday'
                        OR (
                            JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL
                            AND HOUR(order_order.`timestamp`) < 13
                        )
                    )
                    THEN shipping_shippingitem.id
                END) AS "지그재그_당일",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '지그재그 당일배송' AND (
                        JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) = 'nextday'
                        OR (
                            JSON_UNQUOTE(JSON_EXTRACT(order_order.metadata, '$.order_items[0].product.delivery_type')) IS NULL
                            AND HOUR(order_order.`timestamp`) >= 13
                        )
                    )
                    THEN shipping_shippingitem.id
                END) AS "지그재그_이월",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '카카오모빌리티'
                    THEN shipping_shippingitem.id
                END) AS "카카오모빌리티",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '부스터스'
                    THEN shipping_shippingitem.id
                END) AS "부스터스",

                COUNT(DISTINCT CASE
                    WHEN shop_shop.name = '지그재그 당일배송'
                    THEN shipping_shippingitem.id
                END) AS "지그재그 당일배송",

                COUNT(DISTINCT shop_shop.name) AS "셀러수"

            FROM shipping_shippingitem
            JOIN order_order
                ON shipping_shippingitem.order_id = order_order.id
            JOIN hub_cluster_plan_item
                ON shipping_shippingitem.id = hub_cluster_plan_item.shipping_item_id
            JOIN hub_cluster_plan
                ON hub_cluster_plan_item.cluster_plan_id = hub_cluster_plan.id
            JOIN location_address la
                ON shipping_shippingitem.address_id = la.id
            JOIN location_sector ls
                ON shipping_shippingitem.designated_sector_id = ls.id
            JOIN shop_shop
                ON shop_shop.id = order_order.shop_id

            WHERE hub_cluster_plan.plan_date <= CURRENT_DATE()
              AND ls.allowed = 1

            GROUP BY hub_cluster_plan.plan_date
            ORDER BY hub_cluster_plan.plan_date;
        """

        return self.fetch_data(
            cluster_query,
            ssh_host,
            ssh_user,
            ssh_private_key,
            mysql_host,
            mysql_port,
            mysql_user,
            mysql_password,
            mysql_database,
        )


# ==========================================================
# 2. 기존 마스터 데이터 + DB 최신 데이터 병합
# ==========================================================
generator = AutoContainerGeneration(version=7, debug=True)
df_forecast = generator.fetch_cluster_data()

if df_forecast is None or df_forecast.empty:
    print("🚨 DB에서 데이터를 가져오지 못했습니다. 기존 ./data/total_latest.xlsx만 로드합니다.")
    data = pd.read_excel("./data/total_latest.xlsx")
else:
    print("✅ 최신 DB 데이터 추출 완료")

    db_df = df_forecast.copy()
    db_df.columns = [c.replace("총_물량", "총 물량") for c in db_df.columns]
    db_df["날짜"] = pd.to_datetime(db_df["날짜"], errors="coerce").dt.normalize()

    try:
        base = pd.read_excel("./data/total_latest.xlsx")
        base["날짜"] = pd.to_datetime(base["날짜"], errors="coerce").dt.normalize()
    except FileNotFoundError:
        base = pd.DataFrame()

    data = pd.concat([base, db_df], axis=0).reset_index(drop=True)
    data = data.sort_values("날짜")
    data = data.drop_duplicates(subset=["날짜"], keep="last").reset_index(drop=True)


# ==========================================================
# 3. 5-Track 타겟 생성
# ==========================================================
required_cols = [
    "날짜",
    "총 물량",
    "지그재그_당일",
    "지그재그_이월",
    "카카오모빌리티",
    "부스터스",
    "지그재그 당일배송",
    "셀러수",
]

for col in required_cols:
    if col not in data.columns:
        data[col] = 0

data["날짜"] = pd.to_datetime(data["날짜"], errors="coerce").dt.normalize()

numeric_cols = [
    "총 물량",
    "지그재그_당일",
    "지그재그_이월",
    "카카오모빌리티",
    "부스터스",
    "지그재그 당일배송",
    "셀러수",
]

for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce").fillna(0)

# 핵심 변경:
# 순수_기타물량 = 총 물량 - 지그재그_당일 - 지그재그_이월 - 카카오 - 부스터스
data["순수_기타물량"] = (
    data["총 물량"]
    - data["지그재그_당일"]
    - data["지그재그_이월"]
    - data["카카오모빌리티"]
    - data["부스터스"]
).clip(lower=0)

final_cols = [
    "날짜",
    "총 물량",
    "지그재그_당일",
    "지그재그_이월",
    "카카오모빌리티",
    "부스터스",
    "순수_기타물량",
    "지그재그 당일배송",
    "셀러수",
]

data = data[final_cols].sort_values("날짜").reset_index(drop=True)

data.to_excel("./data/total_latest.xlsx", index=False)
print("✨ 5-Track 마스터 데이터 저장 완료")
display(data.tail(10))


# ==========================================================
# 4. 기온 데이터 로드/갱신/병합
# ==========================================================

# 보안상 API 키는 코드에 직접 쓰지 말고 .env에 KMA_API_KEY로 넣는 것을 권장
serviceKey = os.getenv("KMA_API_KEY")

if serviceKey is None:
    print("⚠️ KMA_API_KEY가 .env에 없습니다. 기존 recent_temp.xlsx만 사용합니다.")


def fetch_short_term_forecast(base_date: str, base_time: str, region_point: dict):
    records_all = []

    if serviceKey is None:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    for region, (nx, ny) in tqdm(region_point.items(), desc="단기예보 수집"):
        url = (
            "https://apihub.kma.go.kr/api/typ02/openApi/VilageFcstInfoService_2.0/getVilageFcst"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&base_date={base_date}&base_time={base_time}"
            f"&nx={nx}&ny={ny}&authKey={serviceKey}"
        )

        try:
            response = requests.get(url, verify=False, timeout=20)
            response.raise_for_status()
            json_response = response.json()
            items = json_response["response"]["body"]["items"]["item"]
        except Exception as e:
            print(f"⚠️ {region} 단기 데이터 요청 실패: {e}")
            continue

        informations = {}
        for it in items:
            cate = it["category"]
            fcstDate = it["fcstDate"]
            fcstTime = it["fcstTime"]
            fcstValue = it["fcstValue"]
            key = f"{fcstDate}_{fcstTime}"

            if key not in informations:
                informations[key] = {}
            informations[key][cate] = fcstValue

        region_records = []
        for key, vals in informations.items():
            if "TMP" in vals:
                date = key.split("_")[0]
                region_records.append({"date": date, "TMP": float(vals["TMP"])})

        if not region_records:
            continue

        df_region = (
            pd.DataFrame(region_records)
            .groupby("date", as_index=False)["TMP"]
            .mean()
            .rename(columns={"TMP": "avg_temp"})
        )
        df_region["지역"] = region
        records_all.append(df_region)

    if not records_all:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    df_all = pd.concat(records_all, ignore_index=True)
    df_all.rename(columns={"date": "날짜"}, inplace=True)
    df_all["날짜"] = pd.to_datetime(df_all["날짜"], format="%Y%m%d").dt.strftime("%Y-%m-%d")
    return df_all


def fetch_mid_term_forecast(base_date: str, region_code: dict):
    records = []

    if serviceKey is None:
        return pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

    for region, regId in tqdm(region_code.items(), desc="중기예보 수집"):
        url = (
            "https://apihub.kma.go.kr/api/typ02/openApi/MidFcstInfoService/getMidTa"
            f"?pageNo=1&numOfRows=10000&dataType=JSON"
            f"&regId={regId}&tmFc={base_date}0600&authKey={serviceKey}"
        )

        try:
            response = requests.get(url, timeout=20)
            response.raise_for_status()
            item = response.json()["response"]["body"]["items"]["item"][0]
        except Exception as e:
            print(f"⚠️ {region}({regId}) 중기 데이터 요청 실패: {e}")
            continue

        for day in range(4, 11):
            ta_min = item.get(f"taMin{day}")
            ta_max = item.get(f"taMax{day}")

            if ta_min is not None and ta_max is not None:
                records.append(
                    {
                        "날짜": (datetime.now() + timedelta(days=day)).strftime("%Y-%m-%d"),
                        "지역": region,
                        "avg_temp": (ta_min + ta_max) / 2,
                    }
                )

    return pd.DataFrame(records, columns=["날짜", "지역", "avg_temp"])


region_point = {
    "서울특별시": ("61", "126"),
    "인천광역시": ("54", "125"),
    "경기도": ("60", "121"),
    "대전광역시": ("68", "100"),
    "충청남도": ("65", "110"),
}

region_code = {
    "서울특별시": "11B10101",
    "인천광역시": "11B20201",
    "경기도": "11B20601",
    "대전광역시": "11C20401",
    "충청남도": "11C20101",
}

short_df = fetch_short_term_forecast(
    base_date=datetime.now().strftime("%Y%m%d"),
    base_time="0200",
    region_point=region_point,
)

mid_df = fetch_mid_term_forecast(
    base_date=datetime.now().strftime("%Y%m%d"),
    region_code=region_code,
)

combined_df = pd.concat([short_df, mid_df], ignore_index=True)

if not combined_df.empty:
    combined_df["날짜"] = pd.to_datetime(combined_df["날짜"]).dt.strftime("%Y-%m-%d")
    final_df = combined_df.drop_duplicates(subset=["날짜", "지역"], keep="first").reset_index(drop=True)
    final_df = final_df.sort_values(["지역", "날짜"]).reset_index(drop=True)
    final_df["avg_temp"] = np.round(final_df["avg_temp"], 1)
    final_df = final_df[["날짜", "지역", "avg_temp"]]
else:
    final_df = pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

try:
    base_temp = pd.read_excel("./data/recent_temp.xlsx")
    base_temp["날짜"] = pd.to_datetime(base_temp["날짜"], errors="coerce")
except FileNotFoundError:
    base_temp = pd.DataFrame(columns=["날짜", "지역", "avg_temp"])

temp_data = pd.concat([base_temp, final_df], axis=0).reset_index(drop=True)
temp_data["날짜"] = pd.to_datetime(temp_data["날짜"], errors="coerce")
temp_data = temp_data.drop_duplicates(subset=["날짜", "지역"], keep="last").reset_index(drop=True)
temp_data = temp_data.sort_values(["지역", "날짜"]).reset_index(drop=True)
temp_data.to_excel("./data/recent_temp.xlsx", index=False)

temp_merge_data = temp_data.groupby("날짜", as_index=False)["avg_temp"].mean()
temp_merge_data["avg_temp"] = np.round(temp_merge_data["avg_temp"], 1)
temp_merge_data["날짜"] = pd.to_datetime(temp_merge_data["날짜"], errors="coerce").dt.normalize()

data = pd.merge(data, temp_merge_data, how="left", on="날짜")
print("✅ 기온 데이터 병합 완료")
display(data.tail(10))

AutoContainerGeneration version: 7
Environment: prod-daas-dw-cluster.cluster-cgrjlsurd8vz.ap-northeast-2.rds.amazonaws.com
SSH 터널 연결 성공
쿼리 실행 완료
✅ 최신 DB 데이터 추출 완료
✨ 5-Track 마스터 데이터 저장 완료


,날짜,총 물량,지그재그_당일,지그재그_이월,카카오모빌리티,부스터스,순수_기타물량,지그재그 당일배송,셀러수
843,2026-04-25,12325,6984,5908,374,0,0,12892,3
844,2026-04-26,12676,6923,5450,181,0,122,12373,6
845,2026-04-27,21298,8206,7069,1573,1803,2647,15275,21
846,2026-04-28,19087,6661,7649,944,1836,1997,14310,23
847,2026-04-29,20070,7704,9056,1092,255,1963,16760,22
848,2026-04-30,21014,8574,9804,893,257,1486,18378,22
849,2026-05-01,15064,7124,7453,234,0,253,14577,9
850,2026-05-02,12310,5787,6078,325,0,120,11865,7
851,2026-05-03,10664,5185,5362,0,0,117,10547,4
852,2026-05-04,18711,6033,6732,1811,1181,2954,12765,22


⚠️ KMA_API_KEY가 .env에 없습니다. 기존 recent_temp.xlsx만 사용합니다.
✅ 기온 데이터 병합 완료


,날짜,총 물량,지그재그_당일,지그재그_이월,카카오모빌리티,부스터스,순수_기타물량,지그재그 당일배송,셀러수,avg_temp
843,2026-04-25,12325,6984,5908,374,0,0,12892,3,14.4
844,2026-04-26,12676,6923,5450,181,0,122,12373,6,16.3
845,2026-04-27,21298,8206,7069,1573,1803,2647,15275,21,14.8
846,2026-04-28,19087,6661,7649,944,1836,1997,14310,23,13.3
847,2026-04-29,20070,7704,9056,1092,255,1963,16760,22,13.5
848,2026-04-30,21014,8574,9804,893,257,1486,18378,22,14.4
849,2026-05-01,15064,7124,7453,234,0,253,14577,9,15.1
850,2026-05-02,12310,5787,6078,325,0,120,11865,7,13.2
851,2026-05-03,10664,5185,5362,0,0,117,10547,4,15.4
852,2026-05-04,18711,6033,6732,1811,1181,2954,12765,22,16.0


### Feature & Modeling

In [23]:
# ==========================================================
# Cell 2. Feature + Model 함수 전체
# ==========================================================

import numpy as np
import pandas as pd
import holidays

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_percentage_error


# ==========================================================
# 0. 5-Track 타겟 정의
# ==========================================================
ZIG_SAME = "지그재그_당일"
ZIG_NEXT = "지그재그_이월"
KAKAO = "카카오모빌리티"
BOOSTERS = "부스터스"
OTHER_PURE = "순수_기타물량"
TOTAL = "총 물량"

BASE_TARGETS = [
    ZIG_SAME,
    ZIG_NEXT,
    KAKAO,
    BOOSTERS,
    OTHER_PURE,
]

ALL_TARGETS = BASE_TARGETS + [TOTAL]


# ==========================================================
# 1. 공통 Feature 함수
# ==========================================================
def add_temp_shift_features(df, date_col="날짜", temp_col="avg_temp"):
    df = df.copy().sort_values(date_col)

    # ✅ avg_temp를 반드시 숫자로 변환
    df[temp_col] = pd.to_numeric(df[temp_col], errors="coerce")

    # 결측 보정
    df[temp_col] = (
        df[temp_col]
        .ffill()
        .bfill()
        .fillna(15.0)
    )

    temp_ma7_min = df[temp_col].rolling(7, min_periods=1).min()
    diff = temp_ma7_min - df[temp_col]

    df["temp_shift"] = diff.abs().astype(float)

    # ✅ np.sign 대신 안전한 방식
    df["temp_shift_sign"] = 0
    df.loc[diff > 0, "temp_shift_sign"] = 1
    df.loc[diff < 0, "temp_shift_sign"] = -1

    df[["temp_shift", "temp_shift_sign"]] = (
        df[["temp_shift", "temp_shift_sign"]]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
    )

    return df


def add_target_lag_rolling_features(
    df,
    target_col,
    lags=(1, 7, 14, 21, 28),
    roll_windows=(7, 14, 21, 28),
):
    """
    Target leakage 방지:
    - lag는 shift(lag)
    - rolling은 반드시 shift(1) 이후 계산
    """
    df = df.copy().sort_values("날짜").reset_index(drop=True)

    if target_col not in df.columns:
        for lag in lags:
            df[f"{target_col}_lag_{lag}"] = np.nan
        for w in roll_windows:
            df[f"{target_col}_roll_mean_{w}"] = np.nan
            df[f"{target_col}_roll_std_{w}"] = np.nan
        return df

    s = pd.to_numeric(df[target_col], errors="coerce")

    for lag in lags:
        df[f"{target_col}_lag_{lag}"] = s.shift(lag)

    shifted = s.shift(1)
    for w in roll_windows:
        df[f"{target_col}_roll_mean_{w}"] = shifted.rolling(
            w,
            min_periods=max(2, w // 4),
        ).mean()
        df[f"{target_col}_roll_std_{w}"] = shifted.rolling(
            w,
            min_periods=max(2, w // 4),
        ).std()

    return df


def build_holiday_set(years, manual_holidays=None):
    """
    holidays.KR 자동 공휴일 + 수동 공휴일을 하나의 set(date)로 통합.
    근로자의 날(5/1), 임시공휴일 등은 manual_holidays로 추가.
    """
    kr_holiday_set = set(holidays.KR(years=years).keys())

    if manual_holidays:
        manual_set = set(pd.to_datetime(manual_holidays).date)
        kr_holiday_set.update(manual_set)

    return kr_holiday_set


def add_pre_post_holiday_features(df, holiday_set):
    """
    공휴일 전후 효과를 생성.
    - pre_holiday_peak: 다음 1~2영업일 내 공휴일이 있으면 피크 가능성
    - post_holiday_peak: 직전 공휴일 이후 첫 영업일
    - days_until_holiday: 다음 공휴일까지 남은 일수
    - days_after_holiday: 직전 공휴일로부터 지난 일수
    """
    df = df.copy()
    df_dates = df["날짜"].dt.date.tolist()
    holiday_dates = sorted(list(holiday_set))

    holiday_ts = pd.to_datetime(holiday_dates)

    df["pre_holiday_peak"] = 0
    df["post_holiday_peak"] = 0
    df["days_until_holiday"] = 99
    df["days_after_holiday"] = 99

    if len(holiday_ts) == 0:
        return df

    date_series = df["날짜"]

    for idx, cur_date in enumerate(date_series):
        cur_norm = pd.Timestamp(cur_date).normalize()

        future_holidays = holiday_ts[holiday_ts >= cur_norm]
        past_holidays = holiday_ts[holiday_ts <= cur_norm]

        if len(future_holidays) > 0:
            days_until = int((future_holidays.min() - cur_norm).days)
            df.loc[idx, "days_until_holiday"] = min(days_until, 99)

        if len(past_holidays) > 0:
            days_after = int((cur_norm - past_holidays.max()).days)
            df.loc[idx, "days_after_holiday"] = min(days_after, 99)

    # 기존 v.6 스타일의 pre/post peak도 유지
    holiday_pd = pd.to_datetime(list(holiday_set))

    pre_holiday_map = {}
    for hd in holiday_pd:
        found_days = []
        offset = 1

        while len(found_days) < 2 and offset <= 7:
            prev_day = hd - pd.Timedelta(days=offset)

            if prev_day.weekday() not in [5, 6]:
                found_days.append(prev_day.normalize())

            offset += 1

        if len(found_days) >= 1:
            pre_holiday_map[found_days[0]] = 1
        if len(found_days) >= 2:
            if (found_days[0] - found_days[1]).days == 1:
                pre_holiday_map[found_days[1]] = 2

    post_holiday_map = {}
    for hd in holiday_pd:
        offset = 1

        while offset <= 7:
            next_day = hd + pd.Timedelta(days=offset)

            if next_day.weekday() not in [5, 6]:
                post_holiday_map[next_day.normalize()] = 1
                break

            offset += 1

    df["pre_holiday_peak"] = (
        df["날짜"]
        .map(pre_holiday_map)
        .fillna(0)
        .astype(int)
    )

    df["post_holiday_peak"] = (
        df["날짜"]
        .map(post_holiday_map)
        .fillna(0)
        .astype(int)
    )

    df.loc[df["holiday"] == 1, "pre_holiday_peak"] = 0
    df.loc[df["holiday"] == 1, "post_holiday_peak"] = 0

    return df


def add_seller_closed_features(df, seller_closed_dates=None):
    """
    셀러별 휴무일 feature 생성.
    미래 예측에서 이 값이 1이면 rule-based block 로직이 적용됨.
    """
    df = df.copy()

    seller_to_col = {
        "zigzag": "is_zigzag_closed",
        "kakao": "is_kakao_closed",
        "boosters": "is_boosters_closed",
        "other": "is_other_closed",
    }

    for col in seller_to_col.values():
        df[col] = 0

    if not seller_closed_dates:
        return df

    for seller, dates in seller_closed_dates.items():
        if seller not in seller_to_col:
            continue

        closed_dates = pd.to_datetime(dates).normalize()
        col = seller_to_col[seller]

        df.loc[df["날짜"].isin(closed_dates), col] = 1

    return df


def preprocess_features(
    df,
    manual_holidays=None,
    seller_closed_dates=None,
    manual_spikes_dict=None,
    event_periods=None,
):
    df = df.copy()
    df["날짜"] = pd.to_datetime(df["날짜"], errors="coerce").dt.normalize()
    df = df.sort_values("날짜").reset_index(drop=True)

    # ------------------------------------------------------
    # 순수_기타물량이 없으면 residual로 생성
    # ------------------------------------------------------
    needed_for_other = [TOTAL, ZIG_SAME, ZIG_NEXT, KAKAO, BOOSTERS]

    if OTHER_PURE not in df.columns and all(c in df.columns for c in needed_for_other):
        df[OTHER_PURE] = (
            pd.to_numeric(df[TOTAL], errors="coerce").fillna(0)
            - pd.to_numeric(df[ZIG_SAME], errors="coerce").fillna(0)
            - pd.to_numeric(df[ZIG_NEXT], errors="coerce").fillna(0)
            - pd.to_numeric(df[KAKAO], errors="coerce").fillna(0)
            - pd.to_numeric(df[BOOSTERS], errors="coerce").fillna(0)
        ).clip(lower=0)

    # ------------------------------------------------------
    # 1. 날짜 피처
    # ------------------------------------------------------
    df["month"] = df["날짜"].dt.month
    df["dow"] = df["날짜"].dt.weekday
    df["weekend"] = df["dow"].isin([5, 6]).astype(int)
    df["week_of_year"] = df["날짜"].apply(lambda x: x.isocalendar().week)

    df["month"] = pd.Categorical(df["month"], categories=list(range(1, 13)))
    df["dow"] = pd.Categorical(df["dow"], categories=list(range(0, 7)))

    month_dummies = pd.get_dummies(df["month"], prefix="month").astype(int)
    dow_dummies = pd.get_dummies(df["dow"], prefix="dow").astype(int)

    df = pd.concat([df, month_dummies, dow_dummies], axis=1)

    df["month"] = df["month"].astype(int)
    df["dow"] = df["dow"].astype(int)

    # ------------------------------------------------------
    # 2. 이벤트 피처
    # ------------------------------------------------------
    if event_periods is None:
        event_periods = [
            ("2024-03-25", "2024-04-07"),
            ("2024-06-10", "2024-06-17"),
            ("2024-06-24", "2024-07-07"),
            ("2024-09-23", "2024-10-06"),
            ("2024-11-18", "2024-12-01"),
            ("2025-03-24", "2025-04-07"),
            ("2025-06-01", "2025-06-30"),
            ("2025-09-22", "2025-10-05"),
            ("2025-10-21", "2025-10-21"),
            ("2025-11-17", "2025-12-02"),
            ("2026-03-23", "2026-04-06"),
        ]

    df["is_event"] = 0

    for start, end in event_periods:
        start_dt = pd.to_datetime(start)
        end_dt = pd.to_datetime(end)
        mask = (df["날짜"] >= start_dt) & (df["날짜"] <= end_dt)
        df.loc[mask, "is_event"] = 1

    # 이벤트 block/day index
    df["event_day_index"] = 0
    df["event_block_id"] = 0

    is_event = df["is_event"].eq(1)
    block_id = is_event.ne(is_event.shift()).cumsum()

    df.loc[is_event, "event_block_id"] = block_id[is_event].astype(int)
    df.loc[is_event, "event_day_index"] = (
        df[is_event]
        .groupby(block_id[is_event])
        .cumcount()
        + 1
    )

    df["event_after_1d"] = df["is_event"].shift(1).fillna(0).astype(int)
    df["event_after_2d"] = df["is_event"].shift(2).fillna(0).astype(int)
    df["event_before_1d"] = df["is_event"].shift(-1).fillna(0).astype(int)

    # ------------------------------------------------------
    # 3. 공휴일 피처
    # ------------------------------------------------------
    years = sorted(df["날짜"].dt.year.dropna().unique().tolist())
    holiday_set = build_holiday_set(years, manual_holidays=manual_holidays)

    # Timestamp vs date 타입 문제 방지: 반드시 .dt.date로 비교
    df["holiday"] = df["날짜"].dt.date.isin(holiday_set).astype(int)

    # 주말은 weekend로 따로 잡히므로 holiday에서는 제거
    df.loc[df["weekend"] == 1, "holiday"] = 0

    df = add_pre_post_holiday_features(df, holiday_set)

    # ------------------------------------------------------
    # 4. 셀러별 휴무 피처
    # ------------------------------------------------------
    df = add_seller_closed_features(
        df,
        seller_closed_dates=seller_closed_dates,
    )

    # ------------------------------------------------------
    # 5. 자동 spike 탐지
    # ------------------------------------------------------
    auto_cols = [c for c in ALL_TARGETS if c in df.columns]

    for col in auto_cols:
        df[f"{col}_spike_up_mild"] = 0
        df[f"{col}_spike_up_strong"] = 0
        df[f"{col}_spike_down"] = 0

        trend_col = f"{col}_trend"
        resid_col = f"{col}_resid_ratio"

        df[trend_col] = np.nan
        df[resid_col] = np.nan

        for d in range(7):
            idx_d = df.index[df["dow"] == d]
            if len(idx_d) == 0:
                continue

            s = pd.to_numeric(df.loc[idx_d, col], errors="coerce").sort_index()
            baseline = s.shift(1).rolling(window=3, min_periods=2).mean()
            resid = (s - baseline) / (baseline + 1e-5)

            df.loc[idx_d, trend_col] = baseline.astype(float).values
            df.loc[idx_d, resid_col] = resid.astype(float).values

        for wk_flag in [0, 1]:
            if wk_flag == 0:
                upper1, upper2, lower1 = 0.10, 0.20, -0.15
            else:
                upper1, upper2, lower1 = 0.20, 0.25, -0.10

            mask_up_mild = (
                (df[resid_col] >= upper1)
                & (df[resid_col] < upper2)
                & (df["weekend"] == wk_flag)
            )
            mask_up_strong = (
                (df[resid_col] >= upper2)
                & (df["weekend"] == wk_flag)
            )
            mask_down = (
                (df[resid_col] <= lower1)
                & (df["weekend"] == wk_flag)
            )

            df.loc[mask_up_mild, f"{col}_spike_up_mild"] = 1
            df.loc[mask_up_strong, f"{col}_spike_up_strong"] = 1
            df.loc[mask_down, f"{col}_spike_down"] = 1

    # ------------------------------------------------------
    # 6. 수동 spike 지정
    # ------------------------------------------------------
    if manual_spikes_dict:
        for tgt, spikes in manual_spikes_dict.items():
            for spike_type, dates in spikes.items():
                if dates:
                    dts = pd.to_datetime(dates).normalize()
                    spike_col = f"{tgt}_spike_{spike_type}"

                    if spike_col not in df.columns:
                        df[spike_col] = 0

                    df.loc[df["날짜"].isin(dts), spike_col] = 1

    # ------------------------------------------------------
    # 7. 기온 피처
    # ------------------------------------------------------
    if "avg_temp" in df.columns:
        df = add_temp_shift_features(df, temp_col="avg_temp")
        df = df.drop(columns=["avg_temp"], errors="ignore")

    # ------------------------------------------------------
    # 8. 각 타겟 lag/rolling 생성
    # ------------------------------------------------------
    for tgt in ALL_TARGETS:
        df = add_target_lag_rolling_features(
            df,
            target_col=tgt,
            lags=(1, 7, 14, 21, 28),
            roll_windows=(7, 14, 21, 28),
        )

    # trend/resid는 spike 탐지용이므로 학습에서는 제거
    drop_cols = []
    for col in auto_cols:
        drop_cols += [f"{col}_trend", f"{col}_resid_ratio"]

    df = df.drop(columns=drop_cols, errors="ignore")
    return df


# ==========================================================
# 2. XGBoost Helper
# ==========================================================
def weighted_mse_alpha(alpha=1.5):
    """
    과소예측을 더 강하게 벌점 주는 custom objective.
    label - pred > 0이면 실제가 예측보다 큰 것, 즉 과소예측.
    """
    def obj(label, pred):
        residual = label - pred
        grad = np.where(residual > 0, -2 * alpha * residual, -2 * residual)
        hess = np.where(residual > 0, 2 * alpha, 2.0)
        return grad, hess

    return obj


def _ensure_cols(df, cols, fill_value=0.0):
    out = df.copy()
    missing = [c for c in cols if c not in out.columns]

    for c in missing:
        out[c] = fill_value

    return out[cols]


def _valid_target_mask(df, target_col, zero_run_min=3):
    """
    오래 이어지는 0은 실제 휴무/데이터 결측/운영 중단 가능성이 있으므로
    학습에서 제외하기 위한 mask.
    """
    s = pd.to_numeric(df[target_col], errors="coerce")
    base_mask = s.notna() & (s > 0)

    is_zero = s.fillna(0).eq(0)
    group_id = is_zero.ne(is_zero.shift()).cumsum()
    zero_run_len = is_zero.groupby(group_id).transform("sum")

    long_zero_mask = is_zero & (zero_run_len >= zero_run_min)
    return base_mask & (~long_zero_mask)


def _clean_lag_features_from_fake_zero(df, target_cols, zero_run_min=3):
    """
    긴 0 구간이 lag/rolling에 그대로 들어가면 미래 예측을 과하게 낮출 수 있으므로
    feature 생성 전 NaN으로 바꿔 rolling 계산에서 영향 완화.
    """
    out = df.copy()

    for col in target_cols:
        if col not in out.columns:
            continue

        s = pd.to_numeric(out[col], errors="coerce")
        is_zero = s.fillna(0).eq(0)
        group_id = is_zero.ne(is_zero.shift()).cumsum()
        zero_run_len = is_zero.groupby(group_id).transform("sum")

        fake_zero_mask = is_zero & (zero_run_len >= zero_run_min)
        out.loc[fake_zero_mask, col] = np.nan

    return out


def _overwrite_ts_features(row_df, tgt, buf):
    """
    Recursive forecasting에서 미래 row의 lag/rolling을 최신 buffer로 덮어씀.
    """
    row = row_df.copy()

    clean_buf = [
        float(x)
        for x in buf
        if pd.notna(x) and float(x) > 0
    ]

    for lag in (1, 7, 14, 21, 28):
        row[f"{tgt}_lag_{lag}"] = clean_buf[-lag] if len(clean_buf) >= lag else 0.0

    for w in (7, 14, 21, 28):
        tail = clean_buf[-w:]

        row[f"{tgt}_roll_mean_{w}"] = float(np.mean(tail)) if len(tail) >= 2 else 0.0
        row[f"{tgt}_roll_std_{w}"] = float(np.std(tail, ddof=1)) if len(tail) >= 2 else 0.0

    return row


def apply_seller_block_postprocess(row, tgt, pred_i, other_closed_ratio=0.5):
    """
    셀러별 휴무일 강제 block.
    - 지그재그 휴무: 지그재그_당일/이월 = 0
    - 카카오 휴무: 카카오모빌리티 = 0
    - 부스터스 휴무: 부스터스 = 0
    - 기타 휴무: 순수_기타물량은 비율 감소 옵션 적용
    """
    pred_i = float(pred_i)

    if tgt in [ZIG_SAME, ZIG_NEXT]:
        if int(row.get("is_zigzag_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == KAKAO:
        if int(row.get("is_kakao_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == BOOSTERS:
        if int(row.get("is_boosters_closed", pd.Series([0])).iloc[0]) == 1:
            return 0.0

    if tgt == OTHER_PURE:
        if int(row.get("is_other_closed", pd.Series([0])).iloc[0]) == 1:
            return max(0.0, pred_i * other_closed_ratio)

    return max(0.0, pred_i)


def build_feature_dict(full_df):
    """
    Target leakage 방지용 feature set 구성.
    모든 raw target, 당일 집계 컬럼, 로깅용 실적 컬럼 제거.
    단, 해당 target의 lag/rolling/spike는 사용 가능.
    """
    all_cols = full_df.columns.tolist()
    features_dict = {}

    leakage_cols = [
        "날짜",
        TOTAL,
        ZIG_SAME,
        ZIG_NEXT,
        KAKAO,
        BOOSTERS,
        OTHER_PURE,

        # 기존/로깅/당일 집계 컬럼들
        "지그재그 당일배송",
        "other_seller",
        "기타_당일",
        "기타_이월",
        "기타_물량",
        "기타_총물량",
        "당일_물량",
        "이월_물량",
        "셀러수",
    ]

    for tgt in BASE_TARGETS:
        exclude_other_target_features = []

        for other in ALL_TARGETS:
            if other == tgt:
                continue

            exclude_other_target_features += [
                c for c in all_cols
                if c.startswith(f"{other}_lag_")
                or c.startswith(f"{other}_roll_")
                or c.startswith(f"{other}_spike_")
            ]

        features_dict[tgt] = [
            c for c in all_cols
            if c not in leakage_cols + exclude_other_target_features
        ]

    # Meta 모델은 raw target은 금지.
    # 대신 recursive로 만들 수 있는 TOTAL lag/rolling, 일반 feature, base 예측값은 사용.
    meta_leakage_cols = [
        "날짜",
        TOTAL,
        ZIG_SAME,
        ZIG_NEXT,
        KAKAO,
        BOOSTERS,
        OTHER_PURE,
        "지그재그 당일배송",
        "other_seller",
        "기타_당일",
        "기타_이월",
        "기타_물량",
        "기타_총물량",
        "당일_물량",
        "이월_물량",
        "셀러수",
    ]

    # Meta에서 base target의 lag/rolling은 제외하고, TOTAL lag/rolling만 남김
    exclude_base_ts_for_meta = []
    for base_tgt in BASE_TARGETS:
        exclude_base_ts_for_meta += [
            c for c in all_cols
            if c.startswith(f"{base_tgt}_lag_")
            or c.startswith(f"{base_tgt}_roll_")
            or c.startswith(f"{base_tgt}_spike_")
        ]

    features_dict[TOTAL] = [
        c for c in all_cols
        if c not in meta_leakage_cols + exclude_base_ts_for_meta
    ]

    return features_dict


# ==========================================================
# 3. 학습/검증/미래 예측 함수
# ==========================================================
def train_validate_forecast(
    train,
    forecast_days=7,
    valid_days=7,
    temp_future_df=None,
    manual_holidays=None,
    seller_closed_dates=None,
    manual_spikes_train=None,
    manual_spikes_future=None,
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
    other_closed_ratio=0.5,
    debug_future_features=True,
):
    RANDOM_STATE = 42
    MIN_TRAIN_SAMPLES = 30
    ZERO_RUN_MIN = 3

    train_sorted = train.copy()
    train_sorted["날짜"] = pd.to_datetime(train_sorted["날짜"], errors="coerce").dt.normalize()
    train_sorted = train_sorted.sort_values("날짜").reset_index(drop=True)

    # 순수 기타 타겟 재계산
    if all(c in train_sorted.columns for c in [TOTAL, ZIG_SAME, ZIG_NEXT, KAKAO, BOOSTERS]):
        train_sorted[OTHER_PURE] = (
            pd.to_numeric(train_sorted[TOTAL], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[ZIG_SAME], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[ZIG_NEXT], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[KAKAO], errors="coerce").fillna(0)
            - pd.to_numeric(train_sorted[BOOSTERS], errors="coerce").fillna(0)
        ).clip(lower=0)

    train_for_feature = _clean_lag_features_from_fake_zero(
        train_sorted,
        target_cols=BASE_TARGETS,
        zero_run_min=ZERO_RUN_MIN,
    )

    train_feat_all = preprocess_features(
        train_for_feature,
        manual_holidays=manual_holidays,
        seller_closed_dates=seller_closed_dates,
        manual_spikes_dict=manual_spikes_train,
    )

    split_idx = len(train_feat_all) - valid_days
    train_df = train_feat_all.iloc[:split_idx].copy()
    valid_df = train_feat_all.iloc[split_idx:].copy()
    full_df = train_feat_all.copy()

    last_date = train_sorted["날짜"].max()
    future_dates = pd.date_range(
        start=last_date + pd.Timedelta(days=1),
        periods=forecast_days,
    )

    future_df = pd.DataFrame({"날짜": future_dates})

    if temp_future_df is not None:
        temp_future_df = temp_future_df.copy()
        temp_future_df["날짜"] = pd.to_datetime(temp_future_df["날짜"], errors="coerce").dt.normalize()

        future_temp_input = pd.merge(
            future_df,
            temp_future_df[["날짜", "avg_temp"]],
            on="날짜",
            how="left",
        )

        # 예보가 비는 날짜는 앞뒤 값 또는 15도로 대체
        future_temp_input["avg_temp"] = (
            future_temp_input["avg_temp"]
            .ffill()
            .bfill()
            .fillna(15.0)
        )
    else:
        future_temp_input = future_df.copy()
        future_temp_input["avg_temp"] = 15.0

    future_feat_df = preprocess_features(
        future_temp_input,
        manual_holidays=manual_holidays,
        seller_closed_dates=seller_closed_dates,
        manual_spikes_dict=manual_spikes_future,
    )

    future_feat_df = future_feat_df.iloc[-forecast_days:].reset_index(drop=True)

    for tgt in ALL_TARGETS:
        if tgt not in future_feat_df.columns:
            future_feat_df[tgt] = np.nan

    if debug_future_features:
        print("\n=== 미래 feature 체크 ===")
        debug_cols = [
            "날짜",
            "dow",
            "weekend",
            "holiday",
            "pre_holiday_peak",
            "post_holiday_peak",
            "days_until_holiday",
            "days_after_holiday",
            "is_zigzag_closed",
            "is_kakao_closed",
            "is_boosters_closed",
            "is_other_closed",
        ]
        display(future_feat_df[[c for c in debug_cols if c in future_feat_df.columns]])

    features_dict = build_feature_dict(full_df)

    xgb_base_params = {
        "n_estimators": 900,
        "learning_rate": 0.015,
        "max_depth": 3,
        "min_child_weight": 5,
        "gamma": 0.05,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.1,
        "reg_lambda": 3.0,
        "objective": weighted_mse_alpha(alpha=1.6),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    xgb_meta_params = {
        "n_estimators": 500,
        "learning_rate": 0.02,
        "max_depth": 2,
        "min_child_weight": 8,
        "gamma": 0.1,
        "subsample": 1.0,
        "colsample_bytree": 1.0,
        "reg_alpha": 0.2,
        "reg_lambda": 5.0,
        "objective": weighted_mse_alpha(alpha=1.3),
        "random_state": RANDOM_STATE,
        "tree_method": "hist",
        "eval_metric": "rmse",
    }

    trained_models = {}
    pred_train_base = {}
    pred_valid_base = {}
    pred_full_base = {}

    print("\n=== Base 모델 학습 제외 데이터 확인 ===")

    for tgt in BASE_TARGETS:
        train_recent = train_df.tail(base_recent_window).copy()
        full_recent = full_df.tail(base_recent_window).copy()

        X_tr = _ensure_cols(train_recent, features_dict[tgt]).fillna(0.0)
        y_tr = pd.to_numeric(train_recent[tgt], errors="coerce")

        X_val = _ensure_cols(valid_df, features_dict[tgt]).fillna(0.0)

        X_train_all = _ensure_cols(train_df, features_dict[tgt]).fillna(0.0)
        X_full_all = _ensure_cols(full_df, features_dict[tgt]).fillna(0.0)

        X_full_recent = _ensure_cols(full_recent, features_dict[tgt]).fillna(0.0)
        y_full_recent = pd.to_numeric(full_recent[tgt], errors="coerce")

        mask_tr = _valid_target_mask(train_recent, tgt, ZERO_RUN_MIN)
        mask_full = _valid_target_mask(full_recent, tgt, ZERO_RUN_MIN)

        print(
            f"{tgt}: train 최근 {len(train_recent)}건 중 사용 {mask_tr.sum()}건 / 제외 {len(mask_tr) - mask_tr.sum()}건, "
            f"full 최근 {len(full_recent)}건 중 사용 {mask_full.sum()}건 / 제외 {len(mask_full) - mask_full.sum()}건"
        )

        if mask_tr.sum() >= MIN_TRAIN_SAMPLES:
            model = XGBRegressor(**xgb_base_params)
            model.fit(
                X_tr.loc[mask_tr],
                np.log1p(y_tr.loc[mask_tr].astype(float)),
                verbose=False,
            )

            pred_train_base[tgt] = np.clip(np.expm1(model.predict(X_train_all)), 0, None)
            pred_valid_base[tgt] = np.clip(np.expm1(model.predict(X_val)), 0, None)

            model_full = XGBRegressor(**xgb_base_params)
            model_full.fit(
                X_full_recent.loc[mask_full],
                np.log1p(y_full_recent.loc[mask_full].astype(float)),
                verbose=False,
            )

            pred_full_base[tgt] = np.clip(np.expm1(model_full.predict(X_full_all)), 0, None)
            trained_models[tgt] = model_full

        else:
            recent_valid = y_full_recent.loc[mask_full].tail(28)
            mean_val = float(recent_valid.mean()) if len(recent_valid) > 0 else 0.0

            pred_train_base[tgt] = np.full(len(X_train_all), mean_val)
            pred_valid_base[tgt] = np.full(len(X_val), mean_val)
            pred_full_base[tgt] = np.full(len(X_full_all), mean_val)
            trained_models[tgt] = None

    print("====================================\n")

    # ------------------------------------------------------
    # Meta 모델 학습
    # ------------------------------------------------------
    meta_features = features_dict[TOTAL] + [f"pred_{t}" for t in BASE_TARGETS] + ["base_sum_pred"]

    X_tr_meta = _ensure_cols(train_df, features_dict[TOTAL]).fillna(0.0)
    X_val_meta = _ensure_cols(valid_df, features_dict[TOTAL]).fillna(0.0)
    X_full_meta = _ensure_cols(full_df, features_dict[TOTAL]).fillna(0.0)

    for tgt in BASE_TARGETS:
        X_tr_meta[f"pred_{tgt}"] = pred_train_base[tgt]
        X_val_meta[f"pred_{tgt}"] = pred_valid_base[tgt]
        X_full_meta[f"pred_{tgt}"] = pred_full_base[tgt]

    X_tr_meta["base_sum_pred"] = sum(X_tr_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_val_meta["base_sum_pred"] = sum(X_val_meta[f"pred_{t}"] for t in BASE_TARGETS)
    X_full_meta["base_sum_pred"] = sum(X_full_meta[f"pred_{t}"] for t in BASE_TARGETS)

    X_tr_meta = X_tr_meta[meta_features]
    X_val_meta = X_val_meta[meta_features]
    X_full_meta = X_full_meta[meta_features]

    y_tr_meta = pd.to_numeric(train_df[TOTAL], errors="coerce")
    y_full_meta = pd.to_numeric(full_df[TOTAL], errors="coerce")

    train_meta_recent_idx = train_df.tail(meta_recent_window).index
    full_meta_recent_idx = full_df.tail(meta_recent_window).index

    X_tr_meta_recent = X_tr_meta.loc[train_meta_recent_idx]
    y_tr_meta_recent = y_tr_meta.loc[train_meta_recent_idx]
    train_df_meta_recent = train_df.loc[train_meta_recent_idx]

    X_full_meta_recent = X_full_meta.loc[full_meta_recent_idx]
    y_full_meta_recent = y_full_meta.loc[full_meta_recent_idx]
    full_df_meta_recent = full_df.loc[full_meta_recent_idx]

    mask_tr_meta = _valid_target_mask(train_df_meta_recent, TOTAL, ZERO_RUN_MIN)
    mask_full_meta = _valid_target_mask(full_df_meta_recent, TOTAL, ZERO_RUN_MIN)

    print(
        f"{TOTAL}: meta 최근 {len(train_df_meta_recent)}건 중 사용 {mask_tr_meta.sum()}건 / 제외 {len(mask_tr_meta) - mask_tr_meta.sum()}건"
    )

    model_meta = XGBRegressor(**xgb_meta_params)
    model_meta.fit(
        X_tr_meta_recent.loc[mask_tr_meta],
        np.log1p(y_tr_meta_recent.loc[mask_tr_meta].astype(float)),
        verbose=False,
    )

    meta_valid_pred = np.clip(np.expm1(model_meta.predict(X_val_meta)), 0, None)
    base_sum_valid_pred = X_val_meta["base_sum_pred"].values

    pred_valid_total = (
        (1 - meta_blend_weight) * base_sum_valid_pred
        + meta_blend_weight * meta_valid_pred
    )

    model_meta_full = XGBRegressor(**xgb_meta_params)
    model_meta_full.fit(
        X_full_meta_recent.loc[mask_full_meta],
        np.log1p(y_full_meta_recent.loc[mask_full_meta].astype(float)),
        verbose=False,
    )

    trained_models[TOTAL] = model_meta_full

    # ------------------------------------------------------
    # 검증 성능 출력
    # ------------------------------------------------------
    print("\n=== 검증 성능 (MAPE) ===")

    actual_total = pd.to_numeric(valid_df[TOTAL], errors="coerce").values
    mask_total = actual_total > 0

    if mask_total.sum() > 0:
        mape_total = mean_absolute_percentage_error(
            actual_total[mask_total],
            pred_valid_total[mask_total],
        )
        print(f"총 물량: {mape_total:.4f} (정확도 {100 * (1 - mape_total):.2f}%)")

    for tgt in BASE_TARGETS:
        actuals = pd.to_numeric(valid_df[tgt], errors="coerce").values
        preds = pred_valid_base[tgt]
        mask = actuals > 0

        if mask.sum() > 0:
            mape = mean_absolute_percentage_error(actuals[mask], preds[mask])
            print(f"{tgt}: {mape:.4f} (정확도 {100 * (1 - mape):.2f}%)")

    print("========================\n")

    # ------------------------------------------------------
    # 미래 Recursive Forecasting
    # ------------------------------------------------------
    bufs = {
        tgt: pd.to_numeric(full_df[tgt], errors="coerce").tolist()
        for tgt in ALL_TARGETS
    }

    future_preds = {tgt: [] for tgt in ALL_TARGETS}

    for i in range(forecast_days):
        row = future_feat_df.iloc[[i]].copy()

        for tgt in BASE_TARGETS:
            row = _overwrite_ts_features(row, tgt, bufs[tgt])
            X_row = _ensure_cols(row, features_dict[tgt]).fillna(0.0)

            if trained_models[tgt] is not None:
                pred_i = float(np.expm1(trained_models[tgt].predict(X_row))[0])
                pred_i = max(0.0, pred_i)
            else:
                recent_buf = [
                    x
                    for x in bufs[tgt]
                    if pd.notna(x) and float(x) > 0
                ]
                pred_i = float(np.mean(recent_buf[-7:])) if len(recent_buf) > 0 else 0.0

            # 핵심: 셀러별 휴무일 block 후처리
            pred_i = apply_seller_block_postprocess(
                row=row,
                tgt=tgt,
                pred_i=pred_i,
                other_closed_ratio=other_closed_ratio,
            )

            future_preds[tgt].append(pred_i)
            bufs[tgt].append(pred_i)
            row[f"pred_{tgt}"] = pred_i

        base_sum_pred = sum(row[f"pred_{t}"].iloc[0] for t in BASE_TARGETS)
        row["base_sum_pred"] = base_sum_pred

        row = _overwrite_ts_features(row, TOTAL, bufs[TOTAL])
        X_row_meta = _ensure_cols(row, meta_features).fillna(0.0)

        meta_pred = max(
            0.0,
            float(np.expm1(trained_models[TOTAL].predict(X_row_meta))[0]),
        )

        pred_total = (
            (1 - meta_blend_weight) * base_sum_pred
            + meta_blend_weight * meta_pred
        )

        future_preds[TOTAL].append(pred_total)
        bufs[TOTAL].append(pred_total)

    # ------------------------------------------------------
    # 결과 테이블 구성
    # ------------------------------------------------------
    valid_result_df = pd.DataFrame({
        "날짜": valid_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    valid_result_df["실제_총물량"] = pd.to_numeric(valid_df[TOTAL], errors="coerce").values
    valid_result_df["예측_총물량"] = np.round(pred_valid_total).astype(int)
    valid_result_df["base_sum_예측"] = np.round(base_sum_valid_pred).astype(int)
    valid_result_df["meta_예측"] = np.round(meta_valid_pred).astype(int)

    for tgt in BASE_TARGETS:
        valid_result_df[f"실제_{tgt}"] = pd.to_numeric(valid_df[tgt], errors="coerce").values
        valid_result_df[f"예측_{tgt}"] = np.round(pred_valid_base[tgt]).astype(int)

    forecast_df = pd.DataFrame({
        "날짜": future_feat_df["날짜"].dt.strftime("%Y-%m-%d"),
    })

    for tgt in BASE_TARGETS:
        forecast_df[tgt] = np.round(future_preds[tgt]).astype(int)

    forecast_df["base_sum"] = forecast_df[BASE_TARGETS].sum(axis=1)
    forecast_df["총 물량"] = np.round(future_preds[TOTAL]).astype(int)

    # 휴무일 feature도 결과에 같이 표시해서 block 확인 가능하게 함
    closed_cols = [
        "is_zigzag_closed",
        "is_kakao_closed",
        "is_boosters_closed",
        "is_other_closed",
        "holiday",
        "pre_holiday_peak",
        "post_holiday_peak",
    ]

    for col in closed_cols:
        if col in future_feat_df.columns:
            forecast_df[col] = future_feat_df[col].values

    return trained_models, forecast_df, valid_result_df, features_dict

### Prediction

In [24]:
# ==========================================================
# Cell 3. 실행 코드
# ==========================================================

# 날짜 형식 정리
data["날짜"] = pd.to_datetime(data["날짜"], errors="coerce").dt.normalize()
temp_merge_data["날짜"] = pd.to_datetime(temp_merge_data["날짜"], errors="coerce").dt.normalize()
data["avg_temp"] = pd.to_numeric(data["avg_temp"], errors="coerce")
temp_merge_data["avg_temp"] = pd.to_numeric(temp_merge_data["avg_temp"], errors="coerce")

# ----------------------------------------------------------
# 1. 수동 공휴일
# ----------------------------------------------------------
manual_holidays = [
    "2026-05-01",  # 근로자의 날
]

# ----------------------------------------------------------
# 2. 셀러별 휴무일
# ----------------------------------------------------------
seller_closed_dates = {
    "zigzag": [],   # 지그재그 당일 배송 휴무일
    "kakao": ["2026-05-05", "2026-05-25"], # 카카오 모빌리티 휴무일
    "boosters": ["2026-05-01","2026-05-02","2026-05-05", "2026-05-25"], # 부스터스 휴무일
    "other": ["2026-05-01","2026-05-02","2026-05-05", "2026-05-25"]   # 그 외 셀러들 휴무일(과반 이상 휴무면, 휴무로 태그 넣기)
}

# ----------------------------------------------------------
# 3. 수동 spike
# ----------------------------------------------------------
manual_spikes_future = {
    TOTAL: {
        "down": [],
    },
    ZIG_SAME: {
        "down": [],
    },
    ZIG_NEXT: {
        "down": [],
    },
    KAKAO: {
        "down": [],
    },
    BOOSTERS: {
        "down": [],
    },
    OTHER_PURE: {
        "down": [],
    },
}

# 학습 데이터에는 특별히 강제할 spike가 없으면 None
manual_spikes_train = None

# ----------------------------------------------------------
# 4. 모델 실행
# ----------------------------------------------------------
trained_models, forecast_df, valid_result, features_dict = train_validate_forecast(
    train=data,
    forecast_days=7,
    valid_days=7,
    temp_future_df=temp_merge_data,
    manual_holidays=manual_holidays,
    seller_closed_dates=seller_closed_dates,
    manual_spikes_train=manual_spikes_train,
    manual_spikes_future=manual_spikes_future,
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
    other_closed_ratio=0.5,
    debug_future_features=True,
)

print("\n🔍 [최근 7일 검증 셋 결과 비교표]")
display(valid_result)

print("\n🚀 [미래 7일 예측 결과표]")
display(forecast_df)

print("\n📌 Base 모델별 Feature 개수")
for k, v in features_dict.items():
    print(k, len(v))


=== 미래 feature 체크 ===


,날짜,dow,weekend,holiday,pre_holiday_peak,post_holiday_peak,days_until_holiday,days_after_holiday,is_zigzag_closed,is_kakao_closed,is_boosters_closed,is_other_closed
0,2026-04-29,2,0,0,2,0,2,58,0,0,0,0
1,2026-04-30,3,0,0,1,0,1,59,0,0,0,0
2,2026-05-01,4,0,1,0,0,0,0,0,0,1,1
3,2026-05-02,5,1,0,0,0,3,1,0,0,1,1
4,2026-05-03,6,1,0,0,0,2,2,0,0,0,0
5,2026-05-04,0,0,0,1,1,1,3,0,0,0,0
6,2026-05-05,1,0,1,0,0,0,0,0,1,1,1



=== Base 모델 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 82건 / 제외 68건, full 최근 150건 중 사용 89건 / 제외 61건
지그재그_이월: train 최근 150건 중 사용 82건 / 제외 68건, full 최근 150건 중 사용 89건 / 제외 61건
카카오모빌리티: train 최근 150건 중 사용 148건 / 제외 2건, full 최근 150건 중 사용 148건 / 제외 2건
부스터스: train 최근 150건 중 사용 107건 / 제외 43건, full 최근 150건 중 사용 106건 / 제외 44건
순수_기타물량: train 최근 150건 중 사용 127건 / 제외 23건, full 최근 150건 중 사용 126건 / 제외 24건

총 물량: meta 최근 240건 중 사용 240건 / 제외 0건

=== 검증 성능 (MAPE) ===
총 물량: 0.0445 (정확도 95.55%)
지그재그_당일: 0.0750 (정확도 92.50%)
지그재그_이월: 0.0492 (정확도 95.08%)
카카오모빌리티: 0.1170 (정확도 88.30%)
부스터스: 0.1931 (정확도 80.69%)
순수_기타물량: 0.4499 (정확도 55.01%)


🔍 [최근 7일 검증 셋 결과 비교표]


,날짜,실제_총물량,예측_총물량,base_sum_예측,meta_예측,실제_지그재그_당일,예측_지그재그_당일,실제_지그재그_이월,예측_지그재그_이월,실제_카카오모빌리티,예측_카카오모빌리티,실제_부스터스,예측_부스터스,실제_순수_기타물량,예측_순수_기타물량
840,2026-04-22,17814,19110,19218,18678,6197.0,6757,7333.0,7652,977.0,1165,1089.0,1118,2218.0,2527
841,2026-04-23,19861,19670,19914,18694,7478.0,7163,8721.0,8619,995.0,1169,1095.0,663,1572.0,2300
842,2026-04-24,20536,20194,20457,19139,8121.0,7662,9079.0,8891,1061.0,1091,918.0,942,1357.0,1870
843,2026-04-25,12325,13322,13391,13048,6984.0,6792,5908.0,6146,374.0,323,0.0,3,0.0,127
844,2026-04-26,12676,13503,13703,12703,6923.0,7263,5450.0,5831,181.0,203,0.0,304,122.0,103
845,2026-04-27,21298,21098,21374,19994,6066.0,6966,7292.0,7191,1572.0,1619,1803.0,1730,4565.0,3869
846,2026-04-28,19090,20180,20540,18739,6662.0,7404,7649.0,6543,944.0,818,1836.0,959,1999.0,4817



🚀 [미래 7일 예측 결과표]


,날짜,지그재그_당일,지그재그_이월,카카오모빌리티,부스터스,순수_기타물량,base_sum,총 물량,is_zigzag_closed,is_kakao_closed,is_boosters_closed,is_other_closed,holiday,pre_holiday_peak,post_holiday_peak
0,2026-04-29,7338,8093,1049,634,2175,19289,19182,0,0,0,0,0,2,0
1,2026-04-30,7929,8685,931,355,1941,19841,19711,0,0,0,0,0,1,0
2,2026-05-01,7488,6438,122,0,1366,15414,15789,0,0,1,1,1,0,0
3,2026-05-02,6919,5729,182,0,556,13386,13211,0,0,1,1,0,0,0
4,2026-05-03,7057,5571,138,4,573,13343,13176,0,0,0,0,0,0,0
5,2026-05-04,7329,7235,1344,568,3496,19972,19693,0,0,0,0,0,1,1
6,2026-05-05,7040,5921,0,0,2409,15370,15658,0,1,1,1,1,0,0



📌 Base 모델별 Feature 개수
지그재그_당일 56
지그재그_이월 56
카카오모빌리티 56
부스터스 56
순수_기타물량 56
총 물량 56


# 과거 기반 백테스팅

In [26]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기 (V.7 호환)
# ==========================================================
import numpy as np
import pandas as pd

cutoff_date = pd.to_datetime("2026-04-21")
end_date = pd.to_datetime("2026-04-28")

# --------------------------------------------------------
# 0. 날짜/기온 타입 정리
# --------------------------------------------------------
data["날짜"] = pd.to_datetime(data["날짜"], errors="coerce").dt.normalize()

if "avg_temp" in data.columns:
    data["avg_temp"] = pd.to_numeric(data["avg_temp"], errors="coerce")
else:
    data["avg_temp"] = np.nan

# V.7 핵심 target 보정: 순수_기타물량 없으면 생성
data["순수_기타물량"] = (
    pd.to_numeric(data["총 물량"], errors="coerce").fillna(0)
    - pd.to_numeric(data["지그재그_당일"], errors="coerce").fillna(0)
    - pd.to_numeric(data["지그재그_이월"], errors="coerce").fillna(0)
    - pd.to_numeric(data["카카오모빌리티"], errors="coerce").fillna(0)
    - pd.to_numeric(data["부스터스"], errors="coerce").fillna(0)
).clip(lower=0)

# --------------------------------------------------------
# 1. 4월 21일까지만 학습 데이터로 자르기
# --------------------------------------------------------
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# --------------------------------------------------------
# 2. 4.22 ~ 4.28 실제 기온 데이터 준비
# --------------------------------------------------------
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

future_temp_sim["날짜"] = pd.to_datetime(future_temp_sim["날짜"], errors="coerce").dt.normalize()
future_temp_sim["avg_temp"] = pd.to_numeric(future_temp_sim["avg_temp"], errors="coerce")

# 혹시 기온이 비어 있으면 앞뒤 보정
future_temp_sim["avg_temp"] = (
    future_temp_sim["avg_temp"]
    .ffill()
    .bfill()
    .fillna(15.0)
)

# --------------------------------------------------------
# 3. V.7 수동 공휴일 / 셀러별 휴무일 설정
# --------------------------------------------------------
my_holidays = []  # 예: ["2026-04-25"]

closed_dates_dict = {
    "zigzag": [],
    "kakao": [],
    "boosters": ["2026-04-25", "2026-04-26"],
    "other": [],
}

manual_spikes_future_sim = None
manual_spikes_train_sim = None

# --------------------------------------------------------
# 4. 모델 실행
# --------------------------------------------------------
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim, features_dict_sim = train_validate_forecast(
    train=simulated_data,
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_holidays=my_holidays,              # ✅ 파라미터명 수정
    seller_closed_dates=closed_dates_dict,
    manual_spikes_train=manual_spikes_train_sim,
    manual_spikes_future=manual_spikes_future_sim,
    base_recent_window=150,
    meta_recent_window=240,
    meta_blend_weight=0.2,
    other_closed_ratio=0.5,
    debug_future_features=True,
)

# --------------------------------------------------------
# 5. 실제 정답 가져오기
# --------------------------------------------------------
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# --------------------------------------------------------
# 6. 총 물량 예측 비교
# --------------------------------------------------------
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (총 물량 비교표)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!\n")

# --------------------------------------------------------
# 7. Multi-Track 상세 비교표
# --------------------------------------------------------
print("=" * 60)
print("🔍 [트랙별 상세 백테스팅 결과 확인]")
print("=" * 60)

track_targets = [
    "지그재그_당일",
    "지그재그_이월",
    "카카오모빌리티",
    "부스터스",
    "순수_기타물량",
]

for tgt in track_targets:
    act_tgt = data[
        (data["날짜"] > cutoff_date) &
        (data["날짜"] <= end_date)
    ][["날짜", tgt]].copy()

    act_tgt["날짜"] = act_tgt["날짜"].dt.strftime("%Y-%m-%d")
    act_tgt.rename(columns={tgt: "실제"}, inplace=True)

    pred_tgt = forecast_df_sim[["날짜", tgt]].copy()
    pred_tgt.rename(columns={tgt: "예측"}, inplace=True)

    comp_tgt = pd.merge(act_tgt, pred_tgt, on="날짜", how="left")
    comp_tgt["차이(실제-예측)"] = comp_tgt["실제"] - comp_tgt["예측"]

    comp_tgt["오차율(%)"] = np.where(
        comp_tgt["실제"] > 0,
        (np.abs(comp_tgt["실제"] - comp_tgt["예측"]) / comp_tgt["실제"] * 100).round(2),
        np.nan
    )

    print(f"\n▶️ {tgt} (예측 오차 분석)")
    display(comp_tgt)

# --------------------------------------------------------
# 8. 휴무일/공휴일 feature 확인
# --------------------------------------------------------
debug_cols = [
    "날짜",
    "holiday",
    "is_zigzag_closed",
    "is_kakao_closed",
    "is_boosters_closed",
    "is_other_closed",
    "base_sum",
    "총 물량",
]

print("\n🧩 [예측 결과 내 휴무일/공휴일 반영 확인]")
display(forecast_df_sim[[c for c in debug_cols if c in forecast_df_sim.columns]])

🕒 타임머신 작동: 2026-04-21 기준으로 학습을 시작합니다...


=== 미래 feature 체크 ===


,날짜,dow,weekend,holiday,pre_holiday_peak,post_holiday_peak,days_until_holiday,days_after_holiday,is_zigzag_closed,is_kakao_closed,is_boosters_closed,is_other_closed
0,2026-04-22,2,0,0,0,0,13,51,0,0,0,0
1,2026-04-23,3,0,0,0,0,12,52,0,0,0,0
2,2026-04-24,4,0,0,0,0,11,53,0,0,0,0
3,2026-04-25,5,1,0,0,0,10,54,0,0,1,0
4,2026-04-26,6,1,0,0,0,9,55,0,0,1,0
5,2026-04-27,0,0,0,0,0,8,56,0,0,0,0
6,2026-04-28,1,0,0,0,0,7,57,0,0,0,0



=== Base 모델 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 75건 / 제외 75건, full 최근 150건 중 사용 82건 / 제외 68건
지그재그_이월: train 최근 150건 중 사용 75건 / 제외 75건, full 최근 150건 중 사용 82건 / 제외 68건
카카오모빌리티: train 최근 150건 중 사용 148건 / 제외 2건, full 최근 150건 중 사용 148건 / 제외 2건
부스터스: train 최근 150건 중 사용 109건 / 제외 41건, full 최근 150건 중 사용 107건 / 제외 43건
순수_기타물량: train 최근 150건 중 사용 127건 / 제외 23건, full 최근 150건 중 사용 127건 / 제외 23건

총 물량: meta 최근 240건 중 사용 240건 / 제외 0건

=== 검증 성능 (MAPE) ===
총 물량: 0.0980 (정확도 90.20%)
지그재그_당일: 0.1096 (정확도 89.04%)
지그재그_이월: 0.0866 (정확도 91.34%)
카카오모빌리티: 0.1756 (정확도 82.44%)
부스터스: 0.1783 (정확도 82.17%)
순수_기타물량: 0.8529 (정확도 14.71%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (총 물량 비교표)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-22,20289,17814,13.89,86.11,-2475
1,2026-04-23,21174,19861,6.61,93.39,-1313
2,2026-04-24,20995,20536,2.24,97.76,-459
3,2026-04-25,13368,12325,8.46,91.54,-1043
4,2026-04-26,13174,12676,3.93,96.07,-498
5,2026-04-27,20314,21298,4.62,95.38,984
6,2026-04-28,20724,19090,8.56,91.44,-1634


👉 이 기간의 평균 예측 오차율은 6.90% (정확도 93.10%) 입니다!

🔍 [트랙별 상세 백테스팅 결과 확인]

▶️ 지그재그_당일 (예측 오차 분석)


...

,날짜,실제,예측,차이(실제-예측),오차율(%)
0,2026-04-22,1089,797,292,26.81
1,2026-04-23,1095,828,267,24.38
2,2026-04-24,918,667,251,27.34
3,2026-04-25,0,0,0,NaN
4,2026-04-26,0,0,0,NaN
5,2026-04-27,1803,920,883,48.97
6,2026-04-28,1836,878,958,52.18



▶️ 순수_기타물량 (예측 오차 분석)


,날짜,실제,예측,차이(실제-예측),오차율(%)
0,2026-04-22,2218,2754,-536,24.17
1,2026-04-23,1572,2310,-738,46.95
2,2026-04-24,1357,2850,-1493,110.02
3,2026-04-25,0,182,-182,NaN
4,2026-04-26,122,135,-13,10.66
5,2026-04-27,4565,2904,1661,36.39
6,2026-04-28,1999,3311,-1312,65.63



🧩 [예측 결과 내 휴무일/공휴일 반영 확인]


,날짜,holiday,is_zigzag_closed,is_kakao_closed,is_boosters_closed,is_other_closed,base_sum,총 물량
0,2026-04-22,0,0,0,0,0,20503,20289
1,2026-04-23,0,0,0,0,0,21362,21174
2,2026-04-24,0,0,0,0,0,21199,20995
3,2026-04-25,0,0,0,1,0,13445,13368
4,2026-04-26,0,0,0,1,0,13392,13174
5,2026-04-27,0,0,0,0,0,20591,20314
6,2026-04-28,0,0,0,0,0,21042,20724


In [19]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기 (V.7 호환)
# ==========================================================
import numpy as np
import pandas as pd

cutoff_date = pd.to_datetime("2026-04-14")
end_date = pd.to_datetime("2026-04-21")

data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()

# 1. 4월 21일까지만 학습 데이터로 자르기
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# 2. 4.22 ~ 4.28 (미래 7일)의 실제 기온 데이터만 준비
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

# --------------------------------------------------------
# 💡 [V.7 핵심] 수동 공휴일 및 셀러별 휴무일 시뮬레이션 설정
# (특정 날짜를 넣어 블락(0처리) 효과를 직접 테스트해 볼 수 있습니다)
# --------------------------------------------------------
my_holidays = [] # 예: ["2026-04-25"] (이 날을 강제 휴일로 인식)
closed_dates_dict = {
    "zigzag": [],   # 예: ["2026-04-26"] (이 날짜에 지그재그 예측값 0 강제 보정)
    "kakao": [],
    "boosters": [],
    "other": []
}

# 3. 모델 실행 (타임머신 작동!)
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim = train_validate_forecast(
    train=simulated_data,        # ✅ V.7은 내부에서 전처리하므로 원본을 바로 넣습니다.
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_spikes_train=None,
    manual_spikes_future=None,
    manual_holidays_list=my_holidays,         # ✨ V.7 신규 파라미터 적용
    seller_closed_dates=closed_dates_dict     # ✨ V.7 신규 파라미터 적용
)

# 4. 실제 정답(Ground Truth) 가져오기
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# 5. 예측값과 결합
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

# 6. 오차율 계산
comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (총 물량 비교표)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!\n")


# --------------------------------------------------------
# 7. 💡 [추가] Multi-Track 상세 비교표 (범인 찾기용)
# --------------------------------------------------------
print("="*60)
print("🔍 [트랙별 상세 백테스팅 결과 확인]")
print("="*60)

# other_seller 물량이 원본 data에 없을 경우 대비 (Residual 계산)
if "other_seller" not in data.columns:
    data["other_seller"] = (data["총 물량"].fillna(0) - data.get("지그재그_당일", 0) - data.get("지그재그_이월", 0) - data.get("카카오모빌리티", 0) - data.get("부스터스", 0)).clip(lower=0)

for tgt in ["지그재그_당일", "지그재그_이월", "카카오모빌리티", "부스터스", "other_seller"]:
    act_tgt = data[(data["날짜"] > cutoff_date) & (data["날짜"] <= end_date)][["날짜", tgt]].copy()
    act_tgt["날짜"] = act_tgt["날짜"].dt.strftime("%Y-%m-%d")
    act_tgt.rename(columns={tgt: '실제'}, inplace=True)
    
    pred_tgt = forecast_df_sim[["날짜", tgt]].copy()
    pred_tgt.rename(columns={tgt: '예측'}, inplace=True)
    
    comp_tgt = pd.merge(act_tgt, pred_tgt, on='날짜', how='left')
    comp_tgt['차이(건)'] = comp_tgt['실제'] - comp_tgt['예측']
    
    print(f"\n▶️ {tgt} (예측 오차 분석)")
    display(comp_tgt)

🕒 타임머신 작동: 2026-04-14 기준으로 학습을 시작합니다...


=== 5개의 Base 모델 독립 학습 ===

=== 검증 성능 (MAPE) ===
🌟 총 물량(META): 0.2457 (정확도 75.43%)
✔️ 지그재그_당일: 0.2750 (정확도 72.50%)
✔️ 지그재그_이월: 0.2982 (정확도 70.18%)
✔️ 카카오모빌리티: 0.2020 (정확도 79.80%)
✔️ 부스터스: 0.1788 (정확도 82.12%)
✔️ other_seller: 2.4773 (정확도 -147.73%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (총 물량 비교표)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-15,16305,20900,21.99,78.01,4595
1,2026-04-16,16704,24131,30.78,69.22,7427
2,2026-04-17,16708,22511,25.78,74.22,5803
3,2026-04-18,11912,12410,4.01,95.99,498
4,2026-04-19,10976,12215,10.14,89.86,1239
5,2026-04-20,15259,21091,27.65,72.35,5832
6,2026-04-21,16764,17805,5.85,94.15,1041


👉 이 기간의 평균 예측 오차율은 18.03% (정확도 81.97%) 입니다!

🔍 [트랙별 상세 백테스팅 결과 확인]

▶️ 지그재그_당일 (예측 오차 분석)


,날짜,실제,예측,차이(건)
0,2026-04-15,7672,6259,1413
1,2026-04-16,9285,6199,3086
2,2026-04-17,7369,6029,1340
3,2026-04-18,5793,5266,527
4,2026-04-19,7192,5576,1616
5,2026-04-20,8475,5518,2957
6,2026-04-21,5893,5595,298



▶️ 지그재그_이월 (예측 오차 분석)


...

,날짜,실제,예측,차이(건)
0,2026-04-15,1208,929,279
1,2026-04-16,1251,865,386
2,2026-04-17,996,790,206
3,2026-04-18,407,261,146
4,2026-04-19,171,203,-32
5,2026-04-20,1719,1204,515
6,2026-04-21,1121,912,209



▶️ 부스터스 (예측 오차 분석)


,날짜,실제,예측,차이(건)
0,2026-04-15,1012,704,308
1,2026-04-16,1629,769,860
2,2026-04-17,1016,687,329
3,2026-04-18,0,4,-4
4,2026-04-19,0,11,-11
5,2026-04-20,2006,1140,866
6,2026-04-21,1332,732,600



▶️ other_seller (예측 오차 분석)


,날짜,실제,예측,차이(건)
0,2026-04-15,2120,213,1907
1,2026-04-16,2469,241,2228
2,2026-04-17,2007,186,1821
3,2026-04-18,149,203,-54
4,2026-04-19,117,41,76
5,2026-04-20,3397,79,3318
6,2026-04-21,2535,285,2250


In [59]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기
# ==========================================================
cutoff_date = pd.to_datetime("2026-04-07")
end_date = pd.to_datetime("2026-04-14")

data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()

# 1. 4월 21일까지만 학습
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# 2. 4.22 ~ 4.28 실제 기온만 사용
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

# 3. 모델 실행
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim = train_validate_forecast(
    train=simulated_data,        # ✅ preprocess_features() 넣지 않기
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_spikes_train=None,
    manual_spikes_future=None
)

# 4. 실제 정답 가져오기
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# 5. 예측값과 결합
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

# 6. 오차율 계산
comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!")

🕒 타임머신 작동: 2026-04-07 기준으로 학습을 시작합니다...


=== 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 61건 / 제외 89건, full 최근 150건 중 사용 68건 / 제외 82건
지그재그_이월: train 최근 150건 중 사용 61건 / 제외 89건, full 최근 150건 중 사용 68건 / 제외 82건
기타_당일: train 최근 150건 중 사용 60건 / 제외 90건, full 최근 150건 중 사용 67건 / 제외 83건
기타_이월: train 최근 150건 중 사용 38건 / 제외 112건, full 최근 150건 중 사용 42건 / 제외 108건
총 물량: meta 최근 240건 중 사용 240건 / 제외 0건


=== 검증 성능 (MAPE) ===
총 물량: 0.0911 (정확도 90.89%)
지그재그_당일: 0.0596 (정확도 94.04%)
지그재그_이월: 0.1529 (정확도 84.71%)
기타_당일: 0.2851 (정확도 71.49%)
기타_이월: 5.9124 (정확도 -491.24%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-08,19888,17288,15.04,84.96,-2600
1,2026-04-09,20645,19195,7.55,92.45,-1450
2,2026-04-10,20003,21429,6.65,93.35,1426
3,2026-04-11,14319,12137,17.98,82.02,-2182
4,2026-04-12,13301,11337,17.32,82.68,-1964
5,2026-04-13,19971,20347,1.85,98.15,376
6,2026-04-14,19787,19392,2.04,97.96,-395


👉 이 기간의 평균 예측 오차율은 9.78% (정확도 90.22%) 입니다!


In [60]:
# ==========================================================
# 🕒 백테스팅: 2026-04-21로 돌아가서 4.22 ~ 4.28 예측해보기
# ==========================================================
cutoff_date = pd.to_datetime("2026-03-31")
end_date = pd.to_datetime("2026-04-07")

data["날짜"] = pd.to_datetime(data["날짜"]).dt.normalize()

# 1. 4월 21일까지만 학습
simulated_data = data[data["날짜"] <= cutoff_date].copy()

# 2. 4.22 ~ 4.28 실제 기온만 사용
future_temp_sim = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "avg_temp"]].copy()

# 3. 모델 실행
print(f"🕒 타임머신 작동: {cutoff_date.strftime('%Y-%m-%d')} 기준으로 학습을 시작합니다...\n")

trained_models_sim, forecast_df_sim, valid_result_sim = train_validate_forecast(
    train=simulated_data,        # ✅ preprocess_features() 넣지 않기
    forecast_days=7,
    valid_days=7,
    temp_future_df=future_temp_sim,
    manual_spikes_train=None,
    manual_spikes_future=None
)

# 4. 실제 정답 가져오기
actual_future = data[
    (data["날짜"] > cutoff_date) &
    (data["날짜"] <= end_date)
][["날짜", "총 물량"]].copy()

actual_future.rename(columns={"총 물량": "실제_정답"}, inplace=True)
actual_future["날짜"] = actual_future["날짜"].dt.strftime("%Y-%m-%d")

# 5. 예측값과 결합
comparison_df = pd.merge(
    forecast_df_sim[["날짜", "총 물량"]],
    actual_future,
    on="날짜",
    how="left"
)

comparison_df.rename(columns={"총 물량": "모델_예측값"}, inplace=True)

# 6. 오차율 계산
comparison_df["오차율(%)"] = (
    np.abs(comparison_df["실제_정답"] - comparison_df["모델_예측값"])
    / comparison_df["실제_정답"]
    * 100
).round(2)

comparison_df["정확도(%)"] = (100 - comparison_df["오차율(%)"]).round(2)
comparison_df["차이(실제-예측)"] = comparison_df["실제_정답"] - comparison_df["모델_예측값"]

print("\n🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)")
display(comparison_df)

mean_error = comparison_df["오차율(%)"].mean()
print(f"👉 이 기간의 평균 예측 오차율은 {mean_error:.2f}% (정확도 {100 - mean_error:.2f}%) 입니다!")

🕒 타임머신 작동: 2026-03-31 기준으로 학습을 시작합니다...


=== 학습 제외 데이터 확인 ===
지그재그_당일: train 최근 150건 중 사용 54건 / 제외 96건, full 최근 150건 중 사용 61건 / 제외 89건
지그재그_이월: train 최근 150건 중 사용 54건 / 제외 96건, full 최근 150건 중 사용 61건 / 제외 89건
기타_당일: train 최근 150건 중 사용 53건 / 제외 97건, full 최근 150건 중 사용 60건 / 제외 90건
기타_이월: train 최근 150건 중 사용 33건 / 제외 117건, full 최근 150건 중 사용 38건 / 제외 112건
총 물량: meta 최근 240건 중 사용 240건 / 제외 0건


=== 검증 성능 (MAPE) ===
총 물량: 0.2110 (정확도 78.90%)
지그재그_당일: 0.2120 (정확도 78.80%)
지그재그_이월: 0.2001 (정확도 79.99%)
기타_당일: 0.1771 (정확도 82.29%)
기타_이월: 0.8652 (정확도 13.48%)


🚀 [백테스팅 결과] 4월 22일 ~ 4월 28일 (예측값 vs 실제 정답)


,날짜,모델_예측값,실제_정답,오차율(%),정확도(%),차이(실제-예측)
0,2026-04-01,19635,23192,15.34,84.66,3557
1,2026-04-02,19868,25429,21.87,78.13,5561
2,2026-04-03,20002,24145,17.16,82.84,4143
3,2026-04-04,15346,15320,0.17,99.83,-26
4,2026-04-05,14172,13857,2.27,97.73,-315
5,2026-04-06,19377,22620,14.34,85.66,3243
6,2026-04-07,18504,25511,27.47,72.53,7007


👉 이 기간의 평균 예측 오차율은 14.09% (정확도 85.91%) 입니다!


In [25]:
# 각 트랙별 실제값 vs 예측값 오차 확인하기
tracks = ['지그재그_당일', '지그재그_이월', '기타_당일', '기타_이월']

print("🔍 [트랙별 백테스팅 상세 오차]")
for tgt in tracks:
    # 원본 data에서 실제 정답 가져오기
    actual = data[(data['날짜'] > cutoff_date) & (data['날짜'] <= '2026-04-28')][['날짜', tgt]].copy()
    actual.rename(columns={tgt: '실제'}, inplace=True)
    actual['날짜'] = actual['날짜'].dt.strftime('%Y-%m-%d')
    
    # 예측값 가져오기
    pred = forecast_df_sim[['날짜', tgt]].copy()
    pred.rename(columns={tgt: '예측'}, inplace=True)
    
    comp = pd.merge(actual, pred, on='날짜', how='left')
    comp['차이(건)'] = comp['실제'] - comp['예측']
    
    print(f"\n▶️ {tgt}")
    display(comp)

🔍 [트랙별 백테스팅 상세 오차]

▶️ 지그재그_당일


,날짜,실제,예측,차이(건)
0,2026-04-15,7672,7527.0,145.0
1,2026-04-16,9033,7946.0,1087.0
2,2026-04-17,7369,7509.0,-140.0
3,2026-04-18,7501,5222.0,2279.0
4,2026-04-19,8641,7364.0,1277.0
5,2026-04-20,5708,6416.0,-708.0
6,2026-04-21,5794,6505.0,-711.0
7,2026-04-22,6557,NaN,NaN
8,2026-04-23,7478,NaN,NaN
9,2026-04-24,8121,NaN,NaN



▶️ 지그재그_이월


,날짜,실제,예측,차이(건)
0,2026-04-15,8539,8166.0,373.0
1,2026-04-16,9835,8919.0,916.0
2,2026-04-17,9619,8584.0,1035.0
3,2026-04-18,5881,5688.0,193.0
4,2026-04-19,4746,5045.0,-299.0
5,2026-04-20,5686,7808.0,-2122.0
6,2026-04-21,6873,8391.0,-1518.0
7,2026-04-22,7394,NaN,NaN
8,2026-04-23,8721,NaN,NaN
9,2026-04-24,9079,NaN,NaN



▶️ 기타_당일


,날짜,실제,예측,차이(건)
0,2026-04-15,4680,3126.0,1554.0
1,2026-04-16,4331,3117.0,1214.0
2,2026-04-17,3478,2809.0,669.0
3,2026-04-18,356,449.0,-93.0
4,2026-04-19,171,280.0,-109.0
5,2026-04-20,7217,5418.0,1799.0
6,2026-04-21,3693,3464.0,229.0
7,2026-04-22,3324,NaN,NaN
8,2026-04-23,3655,NaN,NaN
9,2026-04-24,3330,NaN,NaN



▶️ 기타_이월


,날짜,실제,예측,차이(건)
0,2026-04-15,9,11.0,-2.0
1,2026-04-16,10,11.0,-1.0
2,2026-04-17,11,10.0,1.0
3,2026-04-18,0,1.0,-1.0
4,2026-04-19,0,1.0,-1.0
5,2026-04-20,55,24.0,31.0
6,2026-04-21,18,11.0,7.0
7,2026-04-22,13,NaN,NaN
8,2026-04-23,7,NaN,NaN
9,2026-04-24,6,NaN,NaN
